In [1]:
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error,mean_squared_error
import optuna         # for tunning
import xgboost as xgb 
from sklearn.model_selection import KFold

In [2]:
X = pd.read_csv("X_train_42r.csv")
y = pd.read_csv("y_train_42r.csv")
X_test = pd.read_csv("X_test_42r.csv")
y_test = pd.read_csv("y_test_42r.csv")
# X = X.drop(["Latitude [N]","Longitude [E]",'Date'],axis=1)
# X_test = X_test.drop(["Latitude [N]","Longitude [E]",'Date'],axis=1)

In [3]:
from sklearn.preprocessing import StandardScaler

In [4]:
sc = StandardScaler()
X = sc.fit_transform(X)
X_test = sc.transform(X_test)

In [5]:
from sklearn.model_selection import RepeatedKFold

In [6]:
# kfold = RepeatedKFold(n_splits=10,n_repeats=5,random_state=42)
kfold = KFold(n_splits=10)


In [7]:
def objective(trial):
    
    scores = []
    for train_ix, val_ix in kfold.split(X):
        param = { 
        'lambda': trial.suggest_float('lambda', 0, 0.8),
        'alpha': trial.suggest_float('alpha', 0, 0.8),
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),
        "booster": trial.suggest_categorical("booster", ["gbtree"]),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1,1.0),
#         'max_depth' : trial.suggest_int('max_depth', 10, 100, step=1),
#         'min_child_weight' : trial.suggest_int('min_child_weight', 1, 100),
#         'learning_rate': trial.suggest_float('learning_rate',0.0001,0.6),  
#         'gamma' : trial.suggest_float('gamma', 0, 1),
        'n_estimators': trial.suggest_int('n_estimators',100,800,step=2),
        'random_state' : 42}
        
#         if param["booster"] in ["gbtree", "dart"]:
        if param["booster"] in ["gbtree"]:
        # maximum depth of the tree, signifies complexity of the tree.
            param["max_depth"] = trial.suggest_int('max_depth', 50, 500, step=1)
        # minimum child weight, larger the term more conservative the tree.
            param["min_child_weight"] = trial.suggest_int('min_child_weight', 15, 30)
            param["learning_rate"] = trial.suggest_float("learning_rate", 1e-2, 1.0, log=True)
        # defines how selective algorithm is.
            param["gamma"] = trial.suggest_float("gamma", 1e-5, 1e-3, log=True)
#             param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
            param["grow_policy"] = trial.suggest_categorical("grow_policy", ["lossguide"])

#         if param["booster"] == "dart":
#             param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
#             param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
#             param["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0, log=True)
#             param["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0, log=True)
        
        X_train, X_val = X[train_ix], X[val_ix]
        y_train, y_val = y.iloc[train_ix], y.iloc[val_ix]
        
        xgb_model = xgb.XGBRegressor(**param)
       
        xgb_model.fit(X_train, y_train, verbose=False,eval_set=[(X_val,y_val)])
        
        xgb_preds = xgb_model.predict(X_val)
        mae = mean_squared_error(y_val, xgb_preds,squared=False)
        #r2 = r2_score(y_val, xgb_preds)
        scores.append(mae)
        a = np.mean(scores)
        
        return a
    
    
    
    

In [8]:
# study = optuna.create_study(sampler=optuna.samplers.CmaEsSampler(),direction='minimize')
study = optuna.create_study(sampler=optuna.samplers.CmaEsSampler(),direction='minimize')
study.optimize(objective, n_trials=1000,show_progress_bar=True,n_jobs = -1)

[I 2025-04-01 14:53:16,667] A new study created in memory with name: no-name-4fc0ef50-bfae-4999-af94-6123b6fb6cb5
/home/apurva/anaconda3/envs/mlenv/lib/python3.10/site-packages/optuna/progress_bar.py:56: ExperimentalWarning: Progress bar is experimental (supported from v1.2.0). The interface can change in the future.
  self._init_valid()


  0%|          | 0/1000 [00:00<?, ?it/s]

[I 2025-04-01 14:53:17,831] Trial 10 finished with value: 73.94949770546233 and parameters: {'lambda': 0.6088882680193668, 'alpha': 0.020092813766014395, 'subsample': 0.728074214170473, 'booster': 'gbtree', 'colsample_bytree': 0.36168884916098465, 'n_estimators': 146, 'max_depth': 490, 'min_child_weight': 25, 'learning_rate': 0.026363203212637687, 'gamma': 0.0003029474565918844, 'grow_policy': 'lossguide'}. Best is trial 10 with value: 73.94949770546233.
[I 2025-04-01 14:53:17,894] Trial 0 finished with value: 332.26735585767676 and parameters: {'lambda': 0.3053417016479052, 'alpha': 0.13809634547618765, 'subsample': 0.6173229895624721, 'booster': 'gbtree', 'colsample_bytree': 0.7284127108174347, 'n_estimators': 152, 'max_depth': 289, 'min_child_weight': 29, 'learning_rate': 0.012488214053546196, 'gamma': 3.285633380761e-05, 'grow_policy': 'lossguide'}. Best is trial 10 with value: 73.94949770546233.
[I 2025-04-01 14:53:20,195] Trial 1 finished with value: 51.24394288850109 and paramet

[I 2025-04-01 14:53:34,548] Trial 20 finished with value: 54.20407676074417 and parameters: {'lambda': 0.4327098333746558, 'alpha': 0.33141251474910816, 'subsample': 0.6721445029973477, 'booster': 'gbtree', 'colsample_bytree': 0.599760261733329, 'n_estimators': 522, 'max_depth': 380, 'min_child_weight': 21, 'learning_rate': 0.35845361287308375, 'gamma': 0.00010398859599951288, 'grow_policy': 'lossguide'}. Best is trial 3 with value: 38.727049180779986.
[I 2025-04-01 14:53:35,756] Trial 23 finished with value: 53.167520176415415 and parameters: {'lambda': 0.34976663470440017, 'alpha': 0.2761510845812111, 'subsample': 0.5961069696253241, 'booster': 'gbtree', 'colsample_bytree': 0.5405062224537891, 'n_estimators': 364, 'max_depth': 194, 'min_child_weight': 20, 'learning_rate': 0.34635009724293103, 'gamma': 9.152319934072848e-05, 'grow_policy': 'lossguide'}. Best is trial 3 with value: 38.727049180779986.
[I 2025-04-01 14:53:36,130] Trial 26 finished with value: 48.58296998102445 and param

[I 2025-04-01 14:53:46,344] Trial 39 finished with value: 39.72771386207815 and parameters: {'lambda': 0.017315933810927577, 'alpha': 0.30023169894309754, 'subsample': 0.687502411004564, 'booster': 'gbtree', 'colsample_bytree': 0.9616999706226844, 'n_estimators': 442, 'max_depth': 196, 'min_child_weight': 26, 'learning_rate': 0.028601873769589663, 'gamma': 4.986262461723094e-05, 'grow_policy': 'lossguide'}. Best is trial 3 with value: 38.727049180779986.
[I 2025-04-01 14:53:46,874] Trial 47 finished with value: 40.62840550709072 and parameters: {'lambda': 0.4582532920700497, 'alpha': 0.5742169630177174, 'subsample': 0.5722903094542386, 'booster': 'gbtree', 'colsample_bytree': 0.7942921875214097, 'n_estimators': 178, 'max_depth': 235, 'min_child_weight': 24, 'learning_rate': 0.14845617078409057, 'gamma': 2.8134466797401316e-05, 'grow_policy': 'lossguide'}. Best is trial 3 with value: 38.727049180779986.
[I 2025-04-01 14:53:51,248] Trial 45 finished with value: 44.47880822025558 and para

...

[I 2025-04-01 14:59:16,400] Trial 909 finished with value: 38.32571202069571 and parameters: {'lambda': 0.15402992395601364, 'alpha': 0.1865695477766075, 'subsample': 0.765604533775133, 'booster': 'gbtree', 'colsample_bytree': 0.859368337756743, 'n_estimators': 122, 'max_depth': 360, 'min_child_weight': 21, 'learning_rate': 0.060194037874718935, 'gamma': 0.0007219715598096682, 'grow_policy': 'lossguide'}. Best is trial 861 with value: 36.83036377970767.
[I 2025-04-01 14:59:16,490] Trial 907 finished with value: 38.84133898523014 and parameters: {'lambda': 0.1252940446553675, 'alpha': 0.08389172210066922, 'subsample': 0.754647323350562, 'booster': 'gbtree', 'colsample_bytree': 0.8388306237766501, 'n_estimators': 116, 'max_depth': 368, 'min_child_weight': 21, 'learning_rate': 0.07358262416348847, 'gamma': 0.0007664312276941396, 'grow_policy': 'lossguide'}. Best is trial 861 with value: 36.83036377970767.
[I 2025-04-01 14:59:16,562] Trial 912 finished with value: 37.844879319025736 and pa

[I 2025-04-01 14:59:19,216] Trial 925 finished with value: 38.32872301158513 and parameters: {'lambda': 0.1415739611351064, 'alpha': 0.07145246597307904, 'subsample': 0.7673672534672099, 'booster': 'gbtree', 'colsample_bytree': 0.7477159195931055, 'n_estimators': 158, 'max_depth': 382, 'min_child_weight': 22, 'learning_rate': 0.06366210631754543, 'gamma': 0.0006525356900022585, 'grow_policy': 'lossguide'}. Best is trial 861 with value: 36.83036377970767.
[I 2025-04-01 14:59:19,242] Trial 929 finished with value: 37.80788207258367 and parameters: {'lambda': 0.16147967892088996, 'alpha': 0.07196967395719645, 'subsample': 0.7714112167362805, 'booster': 'gbtree', 'colsample_bytree': 0.830721250054098, 'n_estimators': 108, 'max_depth': 388, 'min_child_weight': 22, 'learning_rate': 0.07720789001215643, 'gamma': 0.00033909412017211837, 'grow_policy': 'lossguide'}. Best is trial 861 with value: 36.83036377970767.
[I 2025-04-01 14:59:19,341] Trial 926 finished with value: 38.453988874257945 and

[I 2025-04-01 14:59:22,336] Trial 945 finished with value: 39.47834859303288 and parameters: {'lambda': 0.1744361283563243, 'alpha': 0.03393370364100996, 'subsample': 0.7543950339517383, 'booster': 'gbtree', 'colsample_bytree': 0.8187694323996282, 'n_estimators': 152, 'max_depth': 377, 'min_child_weight': 22, 'learning_rate': 0.07586365722367619, 'gamma': 0.0002663274490048734, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:22,876] Trial 946 finished with value: 38.25691312029403 and parameters: {'lambda': 0.1357677649488754, 'alpha': 0.13261934519357296, 'subsample': 0.7660274523828176, 'booster': 'gbtree', 'colsample_bytree': 0.8258066760825749, 'n_estimators': 162, 'max_depth': 387, 'min_child_weight': 22, 'learning_rate': 0.06584761224118926, 'gamma': 0.0005486167673656691, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:23,522] Trial 950 finished with value: 37.84166506641277 and

[I 2025-04-01 14:59:25,585] Trial 968 finished with value: 38.59794014923688 and parameters: {'lambda': 0.09226409510998898, 'alpha': 0.1609886638783816, 'subsample': 0.768547581551269, 'booster': 'gbtree', 'colsample_bytree': 0.797741296042737, 'n_estimators': 106, 'max_depth': 392, 'min_child_weight': 22, 'learning_rate': 0.07114313943332622, 'gamma': 0.000397766003672786, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:25,704] Trial 961 finished with value: 38.36719202011344 and parameters: {'lambda': 0.09880255384953435, 'alpha': 0.15158956586306974, 'subsample': 0.7791191382305531, 'booster': 'gbtree', 'colsample_bytree': 0.8842947853387384, 'n_estimators': 178, 'max_depth': 372, 'min_child_weight': 21, 'learning_rate': 0.05397047277689019, 'gamma': 0.0005474059793965043, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:25,858] Trial 967 finished with value: 38.62932841503524 and p

[I 2025-04-01 14:59:29,672] Trial 983 finished with value: 38.467051329818304 and parameters: {'lambda': 0.1334248344947972, 'alpha': 0.16755662794122372, 'subsample': 0.784282999969682, 'booster': 'gbtree', 'colsample_bytree': 0.8460976451210293, 'n_estimators': 122, 'max_depth': 355, 'min_child_weight': 22, 'learning_rate': 0.06671927022064879, 'gamma': 0.00041022926461477293, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:30,123] Trial 984 finished with value: 37.836357903611564 and parameters: {'lambda': 0.12282400782737096, 'alpha': 0.19117849341979298, 'subsample': 0.7887971009070869, 'booster': 'gbtree', 'colsample_bytree': 0.8462139887959367, 'n_estimators': 120, 'max_depth': 356, 'min_child_weight': 22, 'learning_rate': 0.06378943240846333, 'gamma': 0.00037820146080890546, 'grow_policy': 'lossguide'}. Best is trial 930 with value: 36.317591602808044.
[I 2025-04-01 14:59:30,321] Trial 982 finished with value: 37.67858964112148

In [10]:
study.best_params

{'lambda': 0.2286636085860371,
 'alpha': 0.11157275959258724,
 'subsample': 0.7743576887387766,
 'booster': 'gbtree',
 'colsample_bytree': 0.9014934822968127,
 'n_estimators': 124,
 'max_depth': 393,
 'min_child_weight': 21,
 'learning_rate': 0.07126324615347485,
 'gamma': 0.0005220723932688918,
 'grow_policy': 'lossguide'}

In [30]:
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [31]:
optuna.visualization.plot_parallel_coordinate(study)

In [32]:
optuna.visualization.plot_slice(study)

In [33]:
optuna.visualization.plot_param_importances(study)